# device check

In [1]:
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


# path

In [2]:
DATA_ROOT = Path(r"D:\dataset\sampled_500")

TRAIN_DIR = DATA_ROOT / "train_mini"
VAL_DIR = DATA_ROOT / "validation"

MODEL_DIR = DATA_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODEL_DIR / "inat_resnet50_best.pth"

#processing

In [3]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# load traning and validation

In [4]:
train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=val_transform
)

assert train_dataset.class_to_idx == val_dataset.class_to_idx, (
    "Train 和 validation 的类别文件夹不一致"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,  # Windows/Jupyter 建议先用 0
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Classes:", len(train_dataset.classes))
print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Class mapping:", train_dataset.class_to_idx)

Classes: 500
Train images: 20000
Validation images: 5000
Class mapping: {'00001_Animalia_Annelida_Polychaeta_Sabellida_Sabellidae_Sabella_spallanzanii': 0, '00003_Animalia_Annelida_Polychaeta_Sabellida_Serpulidae_Spirobranchus_cariniferus': 1, '00018_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Argiope_bruennichi': 2, '00024_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Cyclosa_turbinata': 3, '00038_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Neoscona_crucifera': 4, '00055_Animalia_Arthropoda_Arachnida_Araneae_Filistatidae_Kukulcania_hibernalis': 5, '00128_Animalia_Arthropoda_Arachnida_Araneae_Thomisidae_Synema_globosum': 6, '00145_Animalia_Arthropoda_Arachnida_Opiliones_Phalangiidae_Phalangium_opilio': 7, '00159_Animalia_Arthropoda_Chilopoda_Scolopendromorpha_Scolopendridae_Scolopendra_heros': 8, '00203_Animalia_Arthropoda_Insecta_Coleoptera_Carabidae_Cicindela_hirticollis': 9, '00216_Animalia_Arthropoda_Insecta_Coleoptera_Carabidae_Scaphinotus_angusticollis': 10, '00230_Anim

# load pretrained model

In [6]:
weights = ResNet18_Weights.IMAGENET1K_V1

model = resnet18(weights=weights)

num_classes = len(train_dataset.classes)

model.fc = nn.Linear(
    model.fc.in_features,
    num_classes
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# traning save the best

In [ ]:
EPOCHS = 30
best_val_accuracy = 0.0

for epoch in range(EPOCHS):
    # Training
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_correct += (
            outputs.argmax(dim=1) == labels
        ).sum().item()
        train_total += labels.size(0)

    # Validation
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()
            val_total += labels.size(0)

    train_accuracy = train_correct / train_total
    val_accuracy = val_correct / val_total

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train loss: {train_loss / train_total:.4f} | "
        f"Train acc: {train_accuracy:.4f} | "
        f"Val loss: {val_loss / val_total:.4f} | "
        f"Val acc: {val_accuracy:.4f}"
    )

    num_classes = len(train_dataset.classes)
    model_filename = f"inat_resnet18_{num_classes}classes_imagenet.pth"

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy

        torch.save({
            "model_state_dict": model.state_dict(),
            "classes": train_dataset.classes,
            "num_classes": num_classes
        }, model_filename)

        print(f"Model saved as: {model_filename}")

Epoch 01/10 | Train loss: 5.1075 | Train acc: 0.1244 | Val loss: 3.5701 | Val acc: 0.3152
Model saved as: inat_resnet18_500classes_imagenet.pth
Epoch 02/10 | Train loss: 3.5336 | Train acc: 0.3356 | Val loss: 2.6102 | Val acc: 0.4634
Model saved as: inat_resnet18_500classes_imagenet.pth
Epoch 03/10 | Train loss: 2.8345 | Train acc: 0.4476 | Val loss: 2.1916 | Val acc: 0.5224
Model saved as: inat_resnet18_500classes_imagenet.pth
Epoch 04/10 | Train loss: 2.3979 | Train acc: 0.5184 | Val loss: 1.9389 | Val acc: 0.5702
Model saved as: inat_resnet18_500classes_imagenet.pth
Epoch 05/10 | Train loss: 2.1135 | Train acc: 0.5712 | Val loss: 1.7949 | Val acc: 0.5912
Model saved as: inat_resnet18_500classes_imagenet.pth
Epoch 06/10 | Train loss: 1.8993 | Train acc: 0.6039 | Val loss: 1.7067 | Val acc: 0.6032
Model saved as: inat_resnet18_500classes_imagenet.pth
Epoch 07/10 | Train loss: 1.7388 | Train acc: 0.6318 | Val loss: 1.5911 | Val acc: 0.6272
Model saved as: inat_resnet18_500classes_image

# train with no preset weights

In [11]:
model = resnet18(weights=None)

num_classes = len(train_dataset.classes)

model.fc = nn.Linear(
    model.fc.in_features,
    num_classes
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [13]:
EPOCHS = 30
best_val_accuracy = 0.0

for epoch in range(EPOCHS):
    # Training
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_correct += (
            outputs.argmax(dim=1) == labels
        ).sum().item()
        train_total += labels.size(0)

    # Validation
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()
            val_total += labels.size(0)

    train_accuracy = train_correct / train_total
    val_accuracy = val_correct / val_total

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train loss: {train_loss / train_total:.4f} | "
        f"Train acc: {train_accuracy:.4f} | "
        f"Val loss: {val_loss / val_total:.4f} | "
        f"Val acc: {val_accuracy:.4f}"
    )

    num_classes = len(train_dataset.classes)
    model_filename = f"inat_resnet18_{num_classes}classes_imagenet_no_pretrain_weitght.pth"

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy

        torch.save({
            "model_state_dict": model.state_dict(),
            "classes": train_dataset.classes,
            "num_classes": num_classes
        }, model_filename)

        print(f"Model saved as: {model_filename}")

Epoch 01/30 | Train loss: 4.2485 | Train acc: 0.1474 | Val loss: 4.1137 | Val acc: 0.1640
Model saved as: inat_resnet18_500classes_imagenet_no_pretrain_weitght.pth
Epoch 02/30 | Train loss: 4.1374 | Train acc: 0.1648 | Val loss: 4.1402 | Val acc: 0.1602
Epoch 03/30 | Train loss: 4.0272 | Train acc: 0.1781 | Val loss: 4.0003 | Val acc: 0.1770
Model saved as: inat_resnet18_500classes_imagenet_no_pretrain_weitght.pth
Epoch 04/30 | Train loss: 3.9395 | Train acc: 0.1892 | Val loss: 3.8545 | Val acc: 0.2070
Model saved as: inat_resnet18_500classes_imagenet_no_pretrain_weitght.pth
Epoch 05/30 | Train loss: 3.8514 | Train acc: 0.2000 | Val loss: 3.7770 | Val acc: 0.2168
Model saved as: inat_resnet18_500classes_imagenet_no_pretrain_weitght.pth
Epoch 06/30 | Train loss: 3.7557 | Train acc: 0.2122 | Val loss: 3.7596 | Val acc: 0.2188
Model saved as: inat_resnet18_500classes_imagenet_no_pretrain_weitght.pth
Epoch 07/30 | Train loss: 3.6847 | Train acc: 0.2296 | Val loss: 3.7345 | Val acc: 0.2238
